# Setup 1: Natural Heterogeneity (4-class, IID baseline)
**Settings:** GPU P100 · Persistence: Files only · Internet: ON
**Add Inputs:** Brain Tumor MRI Dataset (masoudnickparvar) + BRISC 2025 (briscdataset)
All-in-one: Data prep → Training (FedAvg + FedProx + QPSO) → Evaluation

In [1]:
import os, sys, copy, time, random, json, shutil
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models, torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from scipy import stats
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42); device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu"); print("CPU only")
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

NUM_CLASSES = 4
CLASS_NAMES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
VALID_CLASSES = {"glioma": 0, "meningioma": 1, "notumor": 2, "no_tumor": 2, "pituitary": 3}

for d in ['data/processed/client1','data/processed/client2','data/processed/client3',
          'data/test_set','models','results/fedavg','results/fedprox','results/qpso','results/plots','logs']:
    os.makedirs(f'/kaggle/working/{d}', exist_ok=True)
print("Directories created")

GPU: Tesla P100-PCIE-16GB
Directories created


## Preprocessor

In [2]:
class Preprocessor:
    def __init__(self, size=(224,224)): self.size = size
    def load_image(self, path):
        try:
            img = Image.open(path)
            if img.mode != "RGB": img = img.convert("RGB")
            return np.array(img.resize(self.size, Image.LANCZOS), dtype=np.float32) / 255.0
        except: return None

    def process_dataset(self, dataset_path, client_name, save_dir="/kaggle/working/data/processed",
                        train_r=0.70, val_r=0.15):
        print(f"\n{'='*60}\nProcessing {client_name} <- {dataset_path}\n{'='*60}")
        images, labels = [], []
        for cls_name in sorted(os.listdir(dataset_path)):
            cls_path = os.path.join(dataset_path, cls_name)
            if not os.path.isdir(cls_path): continue
            key = cls_name.lower().replace(" ","")
            if key not in VALID_CLASSES: print(f"  Skipping: {cls_name}"); continue
            label = VALID_CLASSES[key]
            files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            print(f"  {cls_name}: {len(files)} images (label={label})")
            for fname in tqdm(files, desc=f"    {cls_name}", leave=False):
                arr = self.load_image(os.path.join(cls_path, fname))
                if arr is not None: images.append(arr); labels.append(label)
        X, y = np.array(images), np.array(labels)
        print(f"  Total: {len(X)} | Classes: {np.bincount(y, minlength=NUM_CLASSES)}")
        test_r = 1.0 - train_r - val_r
        X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=test_r, stratify=y, random_state=42)
        val_adj = val_r / (train_r + val_r)
        X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=val_adj, stratify=y_tmp, random_state=42)
        print(f"  Train:{len(X_train)} Val:{len(X_val)} Test:{len(X_test)}")
        out = os.path.join(save_dir, client_name); os.makedirs(out, exist_ok=True)
        for name, arr in [("X_train",X_train),("y_train",y_train),("X_val",X_val),("y_val",y_val),("X_test",X_test),("y_test",y_test)]:
            np.save(os.path.join(out, f"{name}.npy"), arr)
        print(f"  Saved -> {out}")

    @staticmethod
    def create_global_test(pdir="/kaggle/working/data/processed", sdir="/kaggle/working/data/test_set"):
        Xs, ys = [], []
        for i in range(1, 4):
            d = os.path.join(pdir, f"client{i}")
            Xs.append(np.load(os.path.join(d, "X_test.npy")))
            ys.append(np.load(os.path.join(d, "y_test.npy")))
        X, y = np.concatenate(Xs), np.concatenate(ys)
        os.makedirs(sdir, exist_ok=True)
        np.save(os.path.join(sdir, "X_test.npy"), X)
        np.save(os.path.join(sdir, "y_test.npy"), y)
        print(f"Global test: {len(X)} images | Classes: {np.bincount(y, minlength=NUM_CLASSES)}")
print("Preprocessor ready")

Preprocessor ready


## Model & FL Components

In [3]:
class BrainTumorDataset(Dataset):
    _DEFAULT = transforms.Compose([transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __init__(self, X, y, transform=None):
        self.X, self.y = X, y; self.transform = transform or self._DEFAULT
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        img = Image.fromarray((self.X[idx]*255).astype(np.uint8))
        return self.transform(img), torch.tensor(self.y[idx], dtype=torch.long)

TRAIN_TF = transforms.Compose([transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
TEST_TF = transforms.Compose([transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def create_data_loaders(pdir="/kaggle/working/data/processed", tdir="/kaggle/working/data/test_set", bs=32, nw=2):
    loaders = {}
    for i in range(1, 4):
        d = f"{pdir}/client{i}"
        ds_tr = BrainTumorDataset(np.load(f"{d}/X_train.npy"), np.load(f"{d}/y_train.npy"), TRAIN_TF)
        ds_va = BrainTumorDataset(np.load(f"{d}/X_val.npy"), np.load(f"{d}/y_val.npy"), TEST_TF)
        ds_te = BrainTumorDataset(np.load(f"{d}/X_test.npy"), np.load(f"{d}/y_test.npy"), TEST_TF)
        loaders[f"client{i}"] = {"train": DataLoader(ds_tr, bs, shuffle=True, num_workers=nw, pin_memory=True),
            "val": DataLoader(ds_va, bs, shuffle=False, num_workers=nw, pin_memory=True),
            "test": DataLoader(ds_te, bs, shuffle=False, num_workers=nw, pin_memory=True), "train_size": len(ds_tr)}
        print(f"client{i}: Train={len(ds_tr)} Val={len(ds_va)} Test={len(ds_te)}")
    gds = BrainTumorDataset(np.load(f"{tdir}/X_test.npy"), np.load(f"{tdir}/y_test.npy"), TEST_TF)
    loaders["global_test"] = DataLoader(gds, bs, shuffle=False, num_workers=nw, pin_memory=True)
    print(f"Global test: {len(gds)}"); return loaders

class BrainTumorResNet(nn.Module):
    def __init__(self, n=4):
        super().__init__()
        self.model = models.resnet18(weights=None)  # NO pretrained weights — from scratch
        self.model.fc = nn.Linear(self.model.fc.in_features, n)
    def forward(self, x): return self.model(x)

def create_model(n=4, device="cuda"):
    m = BrainTumorResNet(n).to(device)
    print(f"Model: ResNet-18 (from scratch) | Params: {sum(p.numel() for p in m.parameters()):,} | Classes: {n}"); return m

class FederatedClient:
    def __init__(self, cid, train_loader, val_loader, device="cuda"):
        self.client_id = cid; self.train_loader = train_loader
        self.val_loader = val_loader; self.device = device
        self.model = None; self.optimizer = None
        self.criterion = nn.CrossEntropyLoss(); self.dataset_size = len(train_loader.dataset)
    def set_model(self, gm): self.model = copy.deepcopy(gm).to(self.device)
    def set_optimizer(self, lr=0.001): self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
    def train_local(self, epochs=5, verbose=False):
        self.model.train(); el, ea = [], []
        for ep in range(epochs):
            ls, c, t = 0.0, 0, 0
            for imgs, labs in self.train_loader:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                self.optimizer.zero_grad(); out = self.model(imgs)
                loss = self.criterion(out, labs); loss.backward(); self.optimizer.step()
                ls += loss.item(); _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
            el.append(ls/len(self.train_loader)); ea.append(100.*c/t)
        return self.model.state_dict(), el, ea
    def validate(self):
        self.model.eval(); ls, c, t = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labs in self.val_loader:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                out = self.model(imgs); ls += self.criterion(out, labs).item()
                _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
        return ls/len(self.val_loader), 100.*c/t
    def get_dataset_size(self): return self.dataset_size

class FedProxClient(FederatedClient):
    def train_local(self, epochs=5, verbose=False, mu=0.01, global_params=None):
        self.model.train(); el, ea = [], []
        for ep in range(epochs):
            ls, c, t = 0.0, 0, 0
            for imgs, labs in self.train_loader:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                self.optimizer.zero_grad(); out = self.model(imgs)
                loss = self.criterion(out, labs)
                if global_params is not None:
                    prox = sum(torch.norm(pl - pg.detach())**2 for pl, pg in zip(self.model.parameters(), global_params))
                    loss = loss + (mu / 2.0) * prox
                loss.backward(); self.optimizer.step()
                ls += loss.item(); _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
            el.append(ls/len(self.train_loader)); ea.append(100.*c/t)
        return self.model.state_dict(), el, ea

class FedAvgServer:
    def __init__(self, gm, clients, device="cuda"):
        self.global_model = gm; self.clients = clients; self.device = device
        self.total_samples = sum(c.get_dataset_size() for c in clients)
        print(f"FedAvg Server | clients={len(clients)} total={self.total_samples}")
    def aggregate_weights(self, cw):
        first = cw[0][0]; agg = {}
        for k in first:
            if first[k].is_floating_point():
                acc = torch.zeros_like(first[k], dtype=torch.float32)
                for sd, nk in cw: acc = acc + sd[k].float() * (nk / self.total_samples)
                agg[k] = acc.to(first[k].dtype)
            else: agg[k] = first[k].clone()
        return agg
    def evaluate_global_model(self, tl):
        self.global_model.eval(); crit = nn.CrossEntropyLoss(); ls, c, t = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labs in tl:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                out = self.global_model(imgs); ls += crit(out, labs).item()
                _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
        return 100.*c/t, ls/len(tl)
    def get_global_model(self): return copy.deepcopy(self.global_model)

class QPSOServer:
    def __init__(self, gm, clients, device="cuda", beta=0.7):
        self.global_model = gm; self.clients = clients; self.device = device; self.beta = beta
        self.personal_best = {}; self.personal_best_scores = {}
        self.global_best = None; self.global_best_score = 0.0; self.mean_best = None
        print(f"QPSO Server | clients={len(clients)} beta={beta}")
    def initialize_particles(self):
        s = copy.deepcopy(self.global_model.state_dict())
        for c in self.clients: self.personal_best[c.client_id] = copy.deepcopy(s); self.personal_best_scores[c.client_id] = 0.0
        self.global_best = copy.deepcopy(s); self.global_best_score = 0.0; print("QPSO particles initialised")
    def update_personal_best(self, cid, w, acc):
        if acc > self.personal_best_scores[cid]: self.personal_best[cid] = copy.deepcopy(w); self.personal_best_scores[cid] = acc; return True
        return False
    def update_global_best(self, cid, acc):
        if acc > self.global_best_score: self.global_best = copy.deepcopy(self.personal_best[cid]); self.global_best_score = acc; return True
        return False
    def calculate_mean_best(self):
        fid = self.clients[0].client_id
        self.mean_best = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in self.personal_best[fid].items()}
        for c in self.clients:
            for k in self.mean_best: self.mean_best[k] += self.personal_best[c.client_id][k].float()
        for k in self.mean_best: self.mean_best[k] /= len(self.clients)
    def qpso_aggregate(self, cwl):
        for cid, w, acc in cwl:
            pb = self.update_personal_best(cid, w, acc); gb = self.update_global_best(cid, acc)
            if pb: print(f"  {cid}: pbest {acc:.2f}%")
            if gb: print(f"  {cid}: gbest {acc:.2f}%")
        self.calculate_mean_best(); agg = copy.deepcopy(self.global_best)
        for k in agg:
            if not agg[k].is_floating_point(): continue
            ps = torch.zeros_like(agg[k], dtype=torch.float32)
            for c in self.clients: ps += self.personal_best[c.client_id][k].float()
            phi = torch.rand_like(agg[k].float())
            u = torch.rand_like(agg[k].float()).clamp(min=0.3, max=1.0)
            p = phi*(ps/len(self.clients)) + (1-phi)*self.global_best[k].float()
            sign = torch.where(torch.rand_like(agg[k].float())<0.5, torch.ones_like(agg[k].float()), -torch.ones_like(agg[k].float()))
            pert = self.beta * torch.abs(self.mean_best[k].float() - agg[k].float()) * torch.log(1.0/u)
            pert = pert.clamp(-0.1, 0.1)
            agg[k] = (p + sign * pert).to(self.global_best[k].dtype)
        return agg
    def evaluate_global_model(self, tl):
        self.global_model.eval(); crit = nn.CrossEntropyLoss(); ls, c, t = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labs in tl:
                imgs, labs = imgs.to(self.device), labs.to(self.device)
                out = self.global_model(imgs); ls += crit(out, labs).item()
                _, pred = out.max(1); t += labs.size(0); c += pred.eq(labs).sum().item()
        return 100.*c/t, ls/len(tl)
    def get_global_model(self): return copy.deepcopy(self.global_model)
print("All classes loaded")

All classes loaded


## Data Preparation

In [4]:
# Setup 1: Natural Heterogeneity — use datasets as-is with all 4 classes
CLIENT1_PATH = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
CLIENT2_PATH = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/train'
CLIENT3_PATH = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'

# Verify paths
for name, path in [("Client1 (Masoud Test)", CLIENT1_PATH), ("Client2 (BRISC)", CLIENT2_PATH), ("Client3 (Masoud Train)", CLIENT3_PATH)]:
    print(f"\n{name}: {path}")
    if os.path.exists(path):
        for d in sorted(os.listdir(path)):
            full = os.path.join(path, d)
            if os.path.isdir(full): print(f"  {d}/ ({len(os.listdir(full))} files)")
    else: print("  PATH NOT FOUND!")

prep = Preprocessor()
prep.process_dataset(CLIENT1_PATH, 'client1')
prep.process_dataset(CLIENT2_PATH, 'client2')
prep.process_dataset(CLIENT3_PATH, 'client3')
Preprocessor.create_global_test()
data_loaders = create_data_loaders()
print("\nSetup 1 data ready!")


Client1 (Masoud Test): /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
  glioma/ (400 files)
  meningioma/ (400 files)
  notumor/ (400 files)
  pituitary/ (400 files)

Client2 (BRISC): /kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task/train
  glioma/ (1147 files)
  meningioma/ (1329 files)
  no_tumor/ (1067 files)
  pituitary/ (1457 files)

Client3 (Masoud Train): /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
  glioma/ (1400 files)
  meningioma/ (1400 files)
  notumor/ (1400 files)
  pituitary/ (1400 files)

Processing client1 <- /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
  glioma: 400 images (label=0)


    glioma:   0%|          | 0/400 [00:00<?, ?it/s]

  meningioma: 400 images (label=1)


    meningioma:   0%|          | 0/400 [00:00<?, ?it/s]

  notumor: 400 images (label=2)


...

  notumor: 1400 images (label=2)


    notumor:   0%|          | 0/1400 [00:00<?, ?it/s]

  pituitary: 1400 images (label=3)


    pituitary:   0%|          | 0/1400 [00:00<?, ?it/s]

  Total: 5600 | Classes: [1400 1400 1400 1400]
  Train:3919 Val:840 Test:841
  Saved -> /kaggle/working/data/processed/client3
Global test: 1833 images | Classes: [442 470 432 489]
client1: Train=1119 Val=240 Test=241
client2: Train=3499 Val=750 Test=751
client3: Train=3919 Val=840 Test=841
Global test: 1833

Setup 1 data ready!


## Training Functions

In [5]:
def train_fedavg(server, clients, gtl, nr=100, le=5, lr=0.001, se=10, sd="/kaggle/working", verbose=False):
    print("="*80+"\n  FEDAVG TRAINING START\n"+"="*80)
    h = {"round":[],"global_test_acc":[],"global_test_loss":[],"client1_val_acc":[],"client2_val_acc":[],"client3_val_acc":[],"round_time":[]}
    best = 0.0
    for rnd in range(1, nr+1):
        t0 = time.time(); print(f"\n{'='*60} ROUND {rnd}/{nr}"); cw = []
        for cl in clients:
            print(f"  {cl.client_id} training...", end=" ")
            cl.set_model(server.get_global_model()); cl.set_optimizer(lr)
            w,_,_ = cl.train_local(le, verbose=verbose)
            vl, va = cl.validate(); print(f"val={va:.2f}%")
            h[f"{cl.client_id}_val_acc"].append(va); cw.append((w, cl.get_dataset_size()))
        agg = server.aggregate_weights(cw); server.global_model.load_state_dict(agg)
        ga, gl = server.evaluate_global_model(gtl); dt = time.time()-t0
        h["round"].append(rnd); h["global_test_acc"].append(ga); h["global_test_loss"].append(gl); h["round_time"].append(dt)
        print(f"  Global: acc={ga:.2f}% loss={gl:.4f} ({dt:.1f}s)")
        if ga > best: best = ga; torch.save(server.global_model.state_dict(), f"{sd}/models/fedavg_best.pth"); print(f"  New best! {best:.2f}%")
        if rnd % se == 0: torch.save(server.global_model.state_dict(), f"{sd}/models/fedavg_round_{rnd}.pth")
        pd.DataFrame(h).to_csv(f"{sd}/results/fedavg/metrics.csv", index=False)
    print(f"\n{'='*80}\n  FEDAVG DONE best {best:.2f}%\n{'='*80}"); return h

def train_fedprox(server, clients, gtl, nr=100, le=5, lr=0.001, mu=0.01, se=10, sd="/kaggle/working", verbose=False):
    print("="*80+"\n  FEDPROX TRAINING START (mu="+str(mu)+")\n"+"="*80)
    h = {"round":[],"global_test_acc":[],"global_test_loss":[],"client1_val_acc":[],"client2_val_acc":[],"client3_val_acc":[],"round_time":[]}
    best = 0.0
    for rnd in range(1, nr+1):
        t0 = time.time(); print(f"\n{'='*60} ROUND {rnd}/{nr}"); cw = []
        gp = list(server.global_model.parameters())
        for cl in clients:
            print(f"  {cl.client_id} training...", end=" ")
            cl.set_model(server.get_global_model()); cl.set_optimizer(lr)
            w,_,_ = cl.train_local(le, verbose=verbose, mu=mu, global_params=gp)
            vl, va = cl.validate(); print(f"val={va:.2f}%")
            h[f"{cl.client_id}_val_acc"].append(va); cw.append((w, cl.get_dataset_size()))
        agg = server.aggregate_weights(cw); server.global_model.load_state_dict(agg)
        ga, gl = server.evaluate_global_model(gtl); dt = time.time()-t0
        h["round"].append(rnd); h["global_test_acc"].append(ga); h["global_test_loss"].append(gl); h["round_time"].append(dt)
        print(f"  Global: acc={ga:.2f}% loss={gl:.4f} ({dt:.1f}s)")
        if ga > best: best = ga; torch.save(server.global_model.state_dict(), f"{sd}/models/fedprox_best.pth"); print(f"  New best! {best:.2f}%")
        if rnd % se == 0: torch.save(server.global_model.state_dict(), f"{sd}/models/fedprox_round_{rnd}.pth")
        pd.DataFrame(h).to_csv(f"{sd}/results/fedprox/metrics.csv", index=False)
    print(f"\n{'='*80}\n  FEDPROX DONE best {best:.2f}%\n{'='*80}"); return h

def train_qpso(server, clients, gtl, nr=100, le=5, lr=0.001, se=10, sd="/kaggle/working", verbose=False):
    print("="*80+"\n  QPSO-FL TRAINING START\n"+"="*80); server.initialize_particles()
    h = {"round":[],"global_test_acc":[],"global_test_loss":[],"global_best_score":[],
         "client1_val_acc":[],"client2_val_acc":[],"client3_val_acc":[],
         "client1_pbest":[],"client2_pbest":[],"client3_pbest":[],"round_time":[]}
    best = 0.0
    for rnd in range(1, nr+1):
        t0 = time.time(); print(f"\n{'='*60} ROUND {rnd}/{nr}"); cw = []
        for cl in clients:
            print(f"  {cl.client_id} training...", end=" ")
            cl.set_model(server.get_global_model()); cl.set_optimizer(lr)
            w,_,_ = cl.train_local(le, verbose=verbose)
            vl, va = cl.validate(); print(f"val={va:.2f}%")
            h[f"{cl.client_id}_val_acc"].append(va); cw.append((cl.client_id, w, va))
        print("  QPSO aggregating..."); agg = server.qpso_aggregate(cw)
        server.global_model.load_state_dict(agg)
        for c in clients: h[f"{c.client_id}_pbest"].append(server.personal_best_scores[c.client_id])
        h["global_best_score"].append(server.global_best_score)
        ga, gl = server.evaluate_global_model(gtl); dt = time.time()-t0
        h["round"].append(rnd); h["global_test_acc"].append(ga); h["global_test_loss"].append(gl); h["round_time"].append(dt)
        print(f"  Global: acc={ga:.2f}% loss={gl:.4f} gbest={server.global_best_score:.2f}% ({dt:.1f}s)")
        if ga > best: best = ga; torch.save(server.global_model.state_dict(), f"{sd}/models/qpso_best.pth"); print(f"  New best! {best:.2f}%")
        if rnd % se == 0: torch.save(server.global_model.state_dict(), f"{sd}/models/qpso_round_{rnd}.pth")
        pd.DataFrame(h).to_csv(f"{sd}/results/qpso/metrics.csv", index=False)
    print(f"\n{'='*80}\n  QPSO-FL DONE best {best:.2f}% gbest {server.global_best_score:.2f}%\n{'='*80}"); return h
print("Trainers loaded")

Trainers loaded


## FedAvg Training
⚠️ Set `nr=5` for a quick test first!

In [6]:
fedavg_model = create_model(NUM_CLASSES, device)
fedavg_clients = [FederatedClient(f'client{i}', data_loaders[f'client{i}']['train'],
                  data_loaders[f'client{i}']['val'], device=device) for i in range(1, 4)]
fedavg_server = FedAvgServer(fedavg_model, fedavg_clients, device=device)
fedavg_history = train_fedavg(fedavg_server, fedavg_clients, data_loaders['global_test'],
    nr=100, le=5, lr=0.001, se=10, verbose=False)

Model: ResNet-18 (from scratch) | Params: 11,178,564 | Classes: 4
FedAvg Server | clients=3 total=8537
  FEDAVG TRAINING START

============================================================ ROUND 1/100
  client1 training... val=72.08%
  client2 training... val=73.60%
  client3 training... val=47.74%
  Global: acc=49.15% loss=1.4318 (97.8s)
  New best! 49.15%

============================================================ ROUND 2/100
  client1 training... val=77.50%
  client2 training... val=86.93%
  client3 training... val=90.60%
  Global: acc=83.63% loss=0.4029 (97.5s)
  New best! 83.63%

============================================================ ROUND 3/100
  client1 training... val=81.25%
  client2 training... val=86.53%
  client3 training... val=94.76%
  Global: acc=92.42% loss=0.2255 (97.9s)
  New best! 92.42%

============================================================ ROUND 4/100
  client1 training... val=87.08%
  client2 training... val=96.00%
  client3 training... val=91.07%
 

## FedProx Training

In [7]:
torch.cuda.empty_cache()
fedprox_model = create_model(NUM_CLASSES, device)
fedprox_clients = [FedProxClient(f'client{i}', data_loaders[f'client{i}']['train'],
                   data_loaders[f'client{i}']['val'], device=device) for i in range(1, 4)]
fedprox_server = FedAvgServer(fedprox_model, fedprox_clients, device=device)
fedprox_history = train_fedprox(fedprox_server, fedprox_clients, data_loaders['global_test'],
    nr=100, le=5, lr=0.001, mu=0.01, se=10, verbose=False)

Model: ResNet-18 (from scratch) | Params: 11,178,564 | Classes: 4
FedAvg Server | clients=3 total=8537
  FEDPROX TRAINING START (mu=0.01)

============================================================ ROUND 1/100
  client1 training... val=56.25%
  client2 training... val=63.47%
  client3 training... val=51.55%
  Global: acc=25.64% loss=1.6051 (104.9s)
  New best! 25.64%

============================================================ ROUND 2/100
  client1 training... val=31.25%
  client2 training... val=77.73%
  client3 training... val=51.55%
  Global: acc=57.12% loss=1.0560 (104.5s)
  New best! 57.12%

============================================================ ROUND 3/100
  client1 training... val=62.92%
  client2 training... val=72.27%
  client3 training... val=55.60%
  Global: acc=67.65% loss=0.8615 (103.6s)
  New best! 67.65%

============================================================ ROUND 4/100
  client1 training... val=55.42%
  client2 training... val=79.07%
  client3 training..

## QPSO Training

In [8]:
torch.cuda.empty_cache()
qpso_model = create_model(NUM_CLASSES, device)
qpso_clients = [FederatedClient(f'client{i}', data_loaders[f'client{i}']['train'],
                data_loaders[f'client{i}']['val'], device=device) for i in range(1, 4)]
qpso_server = QPSOServer(qpso_model, qpso_clients, device=device, beta=0.7)
qpso_history = train_qpso(qpso_server, qpso_clients, data_loaders['global_test'],
    nr=100, le=5, lr=0.001, se=10, verbose=False)
print("\nAll training complete!")

Model: ResNet-18 (from scratch) | Params: 11,178,564 | Classes: 4
QPSO Server | clients=3 beta=0.7
  QPSO-FL TRAINING START
QPSO particles initialised

============================================================ ROUND 1/100
  client1 training... val=48.33%
  client2 training... val=75.60%
  client3 training... val=89.17%
  QPSO aggregating...
  client1: pbest 48.33%
  client1: gbest 48.33%
  client2: pbest 75.60%
  client2: gbest 75.60%
  client3: pbest 89.17%
  client3: gbest 89.17%
  Global: acc=36.93% loss=1.2158 gbest=89.17% (97.9s)
  New best! 36.93%

============================================================ ROUND 2/100
  client1 training... val=79.17%
  client2 training... val=93.07%
  client3 training... val=93.93%
  QPSO aggregating...
  client1: pbest 79.17%
  client2: pbest 93.07%
  client2: gbest 93.07%
  client3: pbest 93.93%
  client3: gbest 93.93%
  Global: acc=89.74% loss=0.3077 gbest=93.93% (96.3s)
  New best! 89.74%

================================================

## Evaluation & Plots

In [9]:
def plot_accuracy_comparison(dfs, names, save_path):
    n = len(dfs)
    fig, axes = plt.subplots(2, max(n,2), figsize=(8*max(n,2), 12))
    fig.suptitle("FL Aggregation Comparison", fontsize=16, fontweight="bold")
    colors = ['#1f77b4','#ff7f0e','#2ca02c']
    ax = axes[0,0]
    for df, name, c in zip(dfs, names, colors):
        ax.plot(df["round"], df["global_test_acc"], label=name, lw=2, color=c)
    ax.set(xlabel="Round", ylabel="Acc (%)", title="Global Test Accuracy", ylim=[0,100]); ax.legend(); ax.grid(alpha=0.3)
    ax = axes[0,1]
    for df, name, c in zip(dfs, names, colors):
        ax.plot(df["round"], df["global_test_loss"], label=name, lw=2, color=c)
    ax.set(xlabel="Round", ylabel="Loss", title="Global Test Loss"); ax.legend(); ax.grid(alpha=0.3)
    for idx, (df, name) in enumerate(zip(dfs, names)):
        ax = axes[1, idx]
        for i in range(1,4): ax.plot(df["round"], df[f"client{i}_val_acc"], label=f"Client {i}", lw=2)
        ax.set(xlabel="Round", ylabel="Val Acc (%)", title=f"{name}: Per-Client"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()
    print(f"Saved -> {save_path}")

def generate_confusion_matrix(mp, tl, mname, device="cuda", save_path=None):
    model = BrainTumorResNet(NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(mp, map_location=device, weights_only=True))
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for imgs, labs in tl:
            _, pred = model(imgs.to(device)).max(1)
            preds.extend(pred.cpu().numpy()); labels.extend(labs.numpy())
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix - {mname}", fontsize=14, fontweight="bold")
    plt.ylabel("True"); plt.xlabel("Predicted"); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"\n{mname}:"); print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

def plot_roc_auc(mps, mnames, tl, device="cuda", save_path=None):
    n = len(mps)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 6))
    if n == 1: axes = [axes]
    cc = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
    for ax, mp, mn in zip(axes, mps, mnames):
        model = BrainTumorResNet(NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(mp, map_location=device, weights_only=True))
        model.eval(); probs, labs = [], []
        with torch.no_grad():
            for imgs, lb in tl:
                out = model(imgs.to(device)); probs.append(torch.softmax(out, dim=1).cpu().numpy()); labs.extend(lb.numpy())
        yp = np.vstack(probs); yt = np.array(labs)
        # Handle NaN from poorly trained models
        if np.isnan(yp).any():
            print(f"  WARNING: {mn} has NaN outputs, replacing with uniform")
            yp = np.nan_to_num(yp, nan=1.0/NUM_CLASSES)
        yb = label_binarize(yt, classes=list(range(NUM_CLASSES)))
        for i, (cls, c) in enumerate(zip(CLASS_NAMES, cc)):
            try:
                fpr, tpr, _ = roc_curve(yb[:, i], yp[:, i]); a = auc(fpr, tpr)
                ax.plot(fpr, tpr, color=c, lw=2, label=f"{cls} ({a:.4f})")
            except: ax.plot([0,1],[0,1], color=c, lw=2, label=f"{cls} (N/A)")
        try:
            fpr_m, tpr_m, _ = roc_curve(yb.ravel(), yp.ravel())
            ax.plot(fpr_m, tpr_m, 'k--', lw=2, label=f"Micro ({auc(fpr_m,tpr_m):.4f})")
        except: pass
        ax.plot([0,1],[0,1],'gray',ls=':',lw=1)
        ax.set(xlabel="FPR", ylabel="TPR", title=f"ROC - {mn}"); ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show(); print(f"ROC-AUC saved -> {save_path}")

def plot_client_fairness(dfs, names, save_path):
    n = len(dfs)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5)); cc = ["#FF6B6B","#4ECDC4","#45B7D1"]
    if n == 1: axes = [axes]
    for ax, df, t in zip(axes, dfs, names):
        accs = [df[f"client{i}_val_acc"].iloc[-1] for i in range(1,4)]
        ax.bar(["C1","C2","C3"], accs, color=cc, edgecolor="black")
        ax.axhline(np.mean(accs), color="red", ls="--", lw=2, label="Mean")
        ax.set(ylabel="Final Val Acc (%)", ylim=[0,100])
        ax.set_title(f"{t} (std={np.std(accs):.2f}%)", fontsize=13, fontweight="bold"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()

def convergence_metrics(df, target=80.0):
    r = df[df["global_test_acc"]>=target]
    return {"final_acc": round(df["global_test_acc"].iloc[-1],2), "best_acc": round(df["global_test_acc"].max(),2),
            "round_to_target": int(r["round"].min()) if len(r) else None,
            "avg_round_time_s": round(df["round_time"].mean(),2), "total_time_min": round(df["round_time"].sum()/60,2)}

def statistical_analysis(dfa, dfb):
    a, b = dfa["global_test_acc"].values, dfb["global_test_acc"].values
    n = min(len(a),len(b)); a, b = a[:n], b[:n]
    t_stat, p_val = stats.ttest_rel(b, a); d = (b-a).mean()/((b-a).std(ddof=1)+1e-8)
    return {"t_stat": round(float(t_stat),4), "p_value": float(p_val), "cohens_d": round(float(d),4), "significant": bool(p_val<0.05)}

def build_comparison_df(*pairs):
    cols = ["Metric"] + [n for _, n in pairs]
    ml = []
    for df, n in pairs:
        m = convergence_metrics(df)
        m["client_std"] = round(float(np.std([df[f"client{i}_val_acc"].iloc[-1] for i in range(1,4)])),2)
        ml.append(m)
    keys = [("Final Acc (%)","final_acc"),("Best Acc (%)","best_acc"),("Rounds to 80%","round_to_target"),
            ("Avg Round (s)","avg_round_time_s"),("Total Time (min)","total_time_min"),("Client Std Dev","client_std")]
    rows = [[l]+[m[k] if m[k] is not None else "N/A" for m in ml] for l, k in keys]
    return pd.DataFrame(rows, columns=cols)
print("Evaluation functions loaded")

Evaluation functions loaded


In [10]:
df_fa = pd.read_csv('/kaggle/working/results/fedavg/metrics.csv')
df_fp = pd.read_csv('/kaggle/working/results/fedprox/metrics.csv')
df_qp = pd.read_csv('/kaggle/working/results/qpso/metrics.csv')
comp = build_comparison_df((df_fa,"FedAvg"),(df_fp,"FedProx"),(df_qp,"QPSO-FL"))
print(comp.to_string(index=False))
comp.to_csv('/kaggle/working/results/final_comparison.csv', index=False)

          Metric  FedAvg  FedProx  QPSO-FL
   Final Acc (%)   98.75    98.20    98.47
    Best Acc (%)   99.02    98.69    98.58
   Rounds to 80%    2.00     4.00     2.00
   Avg Round (s)   96.26   103.23    96.04
Total Time (min)  160.43   172.04   160.07
  Client Std Dev    1.53     1.44     1.64


In [11]:
plot_accuracy_comparison([df_fa, df_fp, df_qp], ['FedAvg','FedProx','QPSO-FL'], '/kaggle/working/results/plots/comparison.png')
for m, n in [('fedavg','FedAvg'),('fedprox','FedProx'),('qpso','QPSO-FL')]:
    generate_confusion_matrix(f'/kaggle/working/models/{m}_best.pth', data_loaders['global_test'], n, device=device, save_path=f'/kaggle/working/results/plots/cm_{m}.png')
plot_client_fairness([df_fa, df_fp, df_qp], ['FedAvg','FedProx','QPSO-FL'], '/kaggle/working/results/plots/fairness.png')
plot_roc_auc(['/kaggle/working/models/fedavg_best.pth','/kaggle/working/models/fedprox_best.pth','/kaggle/working/models/qpso_best.pth'],
    ['FedAvg','FedProx','QPSO-FL'], data_loaders['global_test'], device=device, save_path='/kaggle/working/results/plots/roc_auc.png')

Saved -> /kaggle/working/results/plots/comparison.png

FedAvg:
              precision    recall  f1-score   support

      Glioma     0.9954    0.9774    0.9863       442
  Meningioma     0.9852    0.9894    0.9873       470
    No Tumor     0.9885    0.9954    0.9919       432
   Pituitary     0.9919    0.9980    0.9949       489

    accuracy                         0.9902      1833
   macro avg     0.9902    0.9900    0.9901      1833
weighted avg     0.9902    0.9902    0.9902      1833


FedProx:
              precision    recall  f1-score   support

      Glioma     0.9908    0.9729    0.9817       442
  Meningioma     0.9850    0.9809    0.9829       470
    No Tumor     0.9839    0.9931    0.9885       432
   Pituitary     0.9879    1.0000    0.9939       489

    accuracy                         0.9869      1833
   macro avg     0.9869    0.9867    0.9868      1833
weighted avg     0.9869    0.9869    0.9869      1833


QPSO-FL:
              precision    recall  f1-score   s

## Statistical Analysis & Summary

In [12]:
s1 = statistical_analysis(df_fa, df_qp)
s2 = statistical_analysis(df_fa, df_fp)
print("FedAvg vs QPSO:", json.dumps(s1, indent=2))
print("FedAvg vs FedProx:", json.dumps(s2, indent=2))
summary = {"fedavg": convergence_metrics(df_fa), "fedprox": convergence_metrics(df_fp),
           "qpso": convergence_metrics(df_qp), "stats_fa_qp": s1, "stats_fa_fp": s2}
with open('/kaggle/working/results/executive_summary.json','w') as f: json.dump(summary, f, indent=2)
print("Summary saved")
print("\nLaTeX Table:")
print(comp.to_latex(index=False, float_format="%.2f", caption="FedAvg vs FedProx vs QPSO-FL", label="tab:comparison"))
print("\nNotebook complete!")

FedAvg vs QPSO: {
  "t_stat": -4.0004,
  "p_value": 0.00012209123699585405,
  "cohens_d": -0.4,
  "significant": true
}
FedAvg vs FedProx: {
  "t_stat": -5.3492,
  "p_value": 5.700683616135362e-07,
  "cohens_d": -0.5349,
  "significant": true
}
Summary saved

LaTeX Table:
\begin{table}
\caption{FedAvg vs FedProx vs QPSO-FL}
\label{tab:comparison}
\begin{tabular}{lrrr}
\toprule
Metric & FedAvg & FedProx & QPSO-FL \\
\midrule
Final Acc (%) & 98.75 & 98.20 & 98.47 \\
Best Acc (%) & 99.02 & 98.69 & 98.58 \\
Rounds to 80% & 2.00 & 4.00 & 2.00 \\
Avg Round (s) & 96.26 & 103.23 & 96.04 \\
Total Time (min) & 160.43 & 172.04 & 160.07 \\
Client Std Dev & 1.53 & 1.44 & 1.64 \\
\bottomrule
\end{tabular}
\end{table}


Notebook complete!
